# Session 18 · Homework — SOLUTIONS (teacher)

Worked solutions with commentary. All cells run top to bottom.
**Central point:** logistic wins every metric — the most complex model did **not** win.
Grade the recommendation on *reasoning* (numbers + interpretability + stakeholders), not
on which model the student names.

## Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score

fraud = pd.read_csv("../../../datasets/secondary/fraud_transactions.csv")
features = ["amount","hour_of_day","is_online","distance_from_home_km","transactions_last_24h"]
X, y = fraud[features], fraud['is_fraud']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)
print('test transactions:', len(y_test), ' fraud:', int(y_test.sum()))

## Part 1 · Comparison table — SOLUTION

tree 0.973/0.688/0.611 · forest 0.978/0.786/0.611 · **logistic 0.984/0.923/0.667**.
Logistic wins accuracy, precision, AND recall. The most complex model (forest) came
second — on this fairly *linear* fraud signal, a linear model fits best.

In [ ]:
# Three contenders. Only logistic needs scaling (trees are scale-invariant),
# so we wrap it in a pipeline that scales first.
tree     = DecisionTreeClassifier(random_state=0)
forest   = RandomForestClassifier(n_estimators=100, random_state=0)
logistic = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))

models = {'decision tree': tree, 'random forest': forest, 'logistic regression': logistic}
for m in models.values():
    m.fit(X_train, y_train)
print('all three models fitted on the same split.')

In [ ]:
rows = []
for name, m in models.items():
    p = m.predict(X_test)
    rows.append({'model': name, 'accuracy': round(accuracy_score(y_test,p),3),
                 'precision': round(precision_score(y_test,p),3),
                 'recall': round(recall_score(y_test,p),3)})
table = pd.DataFrame(rows).set_index('model')
print(table.to_string())
print()
for col in ['accuracy','precision','recall']:
    print(f'{col}: winner = {table[col].idxmax()}')
print()
print('Winner: logistic regression, on every metric. Complexity did NOT win.')

## Part 2 · Interpretability — SOLUTION

- **Tree:** e.g. *'if distance_from_home is high, predict fraud'* — a literal rule.
- **Logistic:** strongest driver is **distance_from_home_km (~+3.2)** — far-from-home
  transactions raise the fraud score most; every feature has a signed weight.
- **Forest:** a black box of 100 voting trees; importances only, no single-decision reason.

In [ ]:
shallow = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_train, y_train)
print(export_text(shallow, feature_names=features))
coef = logistic.named_steps['logisticregression'].coef_[0]
weights = pd.Series(coef, index=features).sort_values(key=abs, ascending=False)
print('logistic weights:'); print(weights.round(2).to_string())

## Part 3 · Recommendation — SOLUTION (sample)

> *I recommend the **logistic regression** model for the fraud team. It is the most
> accurate (0.984), most precise (0.923), and has the best recall (0.667) of the three —
> it catches the most fraud while raising the fewest false alarms, so fewer honest
> customers are wrongly frozen. It is also explainable: each feature has a signed weight,
> and the strongest is distance-from-home, so we can tell a declined customer a concrete
> reason — which matters because a customer whose transaction is blocked is owed an
> explanation they can understand and dispute, and a regulator may require one. The random
> forest, despite being the most sophisticated model, scored lower on every metric AND
> can't give a single-decision reason, so it's the wrong choice here. I'd note, though,
> that this fraud signal is fairly linear; on a problem with strong feature interactions
> I would re-run this comparison, and the forest could well win.*

**Acceptable variation:** a student may recommend the **single decision tree** for its
fully readable rules *if* they acknowledge its lower scores and justify the trade by the
explanation requirement. Full marks require: specific numbers from the table, an
interpretability argument with a named stakeholder (the declined customer / regulator),
and a concession that the best model is problem-dependent. A recommendation with no
numbers, or that ignores interpretability, does not pass.